# 06 — Full Benchmark: All Models on T4

Benchmarks all four models with download time, load time, VRAM usage, and generation throughput. Run this when you want a complete picture of what fits and how fast it goes on a free Colab T4.

| Model | Quant | Size | VRAM | tok/s |
|-------|-------|------|------|-------|
| Qwen3-8B | Q4_K_M | 5.03 GB | 5467 MiB | 32.0 |
| Qwen3.6-35B-A3B | APEX Nano | 11.69 GB | ~12 GB | — |
| Gemma 4 26B-A4B | APEX I-Mini | 12.27 GB | ~13 GB | — |
| gpt-oss-20b | MXFP4 | ~13 GB | ~13 GB | — |

## 1. Install dependencies

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
pip install tabulate psutil


## 2. Define the benchmark harness

In [ ]:
import time, os, subprocess
from llama_cpp import Llama
from huggingface_hub import hf_hub_download, list_repo_files
from tabulate import tabulate

BENCH_PROMPT = "Explain the concept of recursion in programming, with an example."
N_TOKENS = 128

def benchmark(model_name, repo_id, filename=None):
    results = {"model": model_name}
    
    # Find filename if not given
    if not filename:
        files = list_repo_files(repo_id)
        ggufs = [f for f in files if f.endswith('.gguf')]
        filename = ggufs[0] if ggufs else None
    if not filename:
        results["error"] = "No GGUF found"
        return results
    
    # Download
    t0 = time.time()
    model_path = hf_hub_download(repo_id=repo_id, filename=filename)
    results["download_s"] = time.time() - t0
    results["size_gb"] = os.path.getsize(model_path) / 1024**3
    
    # Load
    t0 = time.time()
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=4096, verbose=False)
    results["load_s"] = time.time() - t0
    
    # VRAM
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True)
    results["vram_mib"] = int(r.stdout.strip())
    
    # Warmup
    llm(BENCH_PROMPT, max_tokens=16, verbose=False)
    
    # Timed generation
    t0 = time.time()
    llm(BENCH_PROMPT, max_tokens=N_TOKENS, verbose=False)
    results["gen_s"] = time.time() - t0
    results["tok_s"] = N_TOKENS / results["gen_s"]
    
    # Free memory
    del llm
    import gc; gc.collect()
    
    return results

all_results = []


## 3. Benchmark Qwen3-8B Q4_K_M

In [ ]:
r = benchmark("Qwen3-8B Q4_K_M", "Qwen/Qwen3-8B-GGUF", "qwen3-8b-q4_k_m.gguf")
all_results.append(r)
print(r)


## 4. Benchmark Qwen3.6-35B-A3B APEX Nano

In [ ]:
r = benchmark("Qwen3.6-35B APEX Nano", "mudler/Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-GGUF")
all_results.append(r)
print(r)


## 5. Benchmark Gemma 4 26B-A4B APEX I-Mini

In [ ]:
r = benchmark("Gemma4 26B APEX I-Mini", "mudler/Carnice-Gemma4-26B-A4B-APEX-GGUF")
all_results.append(r)
print(r)


## 6. Results table

In [ ]:
rows = []
for r in all_results:
    if "error" in r:
        rows.append([r["model"], "ERROR", "", "", "", ""])
        continue
    rows.append([
        r["model"],
        f"{r['size_gb']:.2f} GB",
        f"{r['vram_mib']} MiB",
        f"{r['load_s']:.1f}s",
        f"{r.get('tok_s', 0):.1f} tok/s",
        f"{r['download_s']:.1f}s",
    ])

print(tabulate(rows,
    headers=["Model", "Size", "VRAM", "Load", "Throughput", "Download"],
    tablefmt="github"))


## Notes

- **T4 free tier:** 15360 MiB VRAM, ~13 GB RAM, 2 vCPU
- All models here are **proven to fit** on a free T4.
- Throughput for MoE models (35B, 26B) will be lower than dense Qwen3-8B because of expert routing overhead, but they fit where a dense 35B/26B never would.
- For faster MoE inference, consider building **ik_llama.cpp** (see README).